# LAB 01

In [3]:
import pygame
import math

LARGURA, ALTURA = 900, 650
FPS = 60

class Robo:
    def __init__(self, x, y):
        self.x = float(x)
        self.y = float(y)
        self.theta = 0.0
        self.angulos = [-math.pi / 4, 0, math.pi / 4]
        self.alcance = 150.0
        self.leituras = [self.alcance] * 3

    def medir(self, obstaculos):
        self.leituras = []

        for beta in self.angulos:
            angulo = self.theta + beta
            distancia = self.alcance

            for passo in range(5, int(self.alcance), 4):
                rx = self.x + passo * math.cos(angulo)
                ry = self.y + passo * math.sin(angulo)

                if rx <= 0 or rx >= LARGURA or ry <= 0 or ry >= ALTURA:
                    distancia = float(passo)
                    break

                bateu = False

                for obstaculo in obstaculos:
                    if obstaculo.collidepoint(rx, ry):
                        distancia = float(passo)
                        bateu = True
                        break

                if bateu:
                    break

            self.leituras.append(distancia)

    def desenhar(self, tela):
        for i, beta in enumerate(self.angulos):
            angulo = self.theta + beta
            distancia = self.leituras[i]

            rx = self.x + distancia * math.cos(angulo)
            ry = self.y + distancia * math.sin(angulo)

            cor = (255, 200, 0) if distancia < self.alcance else (0, 255, 100)

            pygame.draw.line(tela, cor, (int(self.x), int(self.y)), (int(rx), int(ry)), 2)
            pygame.draw.circle(tela, cor, (int(rx), int(ry)), 4)

        pos = (int(self.x), int(self.y))
        pygame.draw.circle(tela, (0, 200, 255), pos, 16)

        fx = self.x + 24 * math.cos(self.theta)
        fy = self.y + 24 * math.sin(self.theta)

        pygame.draw.line(tela, (255, 50, 50), pos, (int(fx), int(fy)), 3)

def main():
    pygame.init()

    tela = pygame.display.set_mode((LARGURA, ALTURA))
    pygame.display.set_caption("LAB 01 - Raycasting")
    relogio = pygame.time.Clock()
    fonte = pygame.font.SysFont("monospace", 14)

    robo = Robo(150, 300)

    obstaculos = [
        pygame.Rect(350, 150, 100, 350),
        pygame.Rect(600, 100, 150, 100),
        pygame.Rect(600, 400, 150, 150)
    ]

    rodando = True

    while rodando:
        relogio.tick(FPS)

        for evento in pygame.event.get():
            if evento.type == pygame.QUIT:
                rodando = False

        mx, my = pygame.mouse.get_pos()

        dx = mx - robo.x
        dy = my - robo.y

        robo.theta = math.atan2(dy, dx)
        robo.x += dx * 0.03
        robo.y += dy * 0.03

        robo.medir(obstaculos)

        tela.fill((20, 24, 30))

        for obstaculo in obstaculos:
            pygame.draw.rect(tela, (180, 50, 50), obstaculo)

        robo.desenhar(tela)

        nomes = ["Esq", "Frente", "Dir"]

        for i, distancia in enumerate(robo.leituras):
            texto = f"{nomes[i]}: {distancia:.1f} px"
            tela.blit(fonte.render(texto, True, (220, 220, 220)), (20, 20 + i * 20))

        pygame.display.flip()

    pygame.quit()

main()